# Phase 7: Real Dataset Integration

This notebook downloads the SARS-CoV-2 reference genome `NC_045512.2`, parses FASTA data, generates fixed-length sliding-window fragments, creates multiple real dataset pair workloads, and benchmarks the existing Hamming Distance CPU and CUDA implementations on real genomic fragments.

If you are not already in the repository root, clone or open the repository first. Example only:

```bash
# !git clone <repository-url>
# %cd CUDA-Bioinformatic
```

In [ ]:
!nvidia-smi

In [ ]:
!nvcc --version

In [ ]:
import os
from pathlib import Path

print("Current working directory:", os.getcwd())
required_paths = [Path("scripts"), Path("src"), Path("benchmarks")]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise RuntimeError(f"This notebook should be run from the repository root. Missing: {missing_paths}")

## Download Dataset

In [ ]:
!python scripts/download_datasets.py \
  --dataset sars-cov-2 \
  --output data/raw/sars_cov_2_NC_045512_2.fasta

## Recompile Encoded GPU Executable

In [ ]:
!nvcc src/hamming_gpu_encoded.cu -O3 -std=c++17 -I src/common -o hamming_gpu_encoded

## Adjacent Mode Benchmark

In [ ]:
!python benchmarks/run_real_dataset_benchmark.py \
  --window-size 128 \
  --stride 32 \
  --pairing-mode adjacent \
  --repetitions 5

## Sampled Mode Benchmark

In [ ]:
!python benchmarks/run_real_dataset_benchmark.py \
  --window-size 128 \
  --stride 32 \
  --pairing-mode sampled \
  --pairs-per-fragment 64 \
  --max-pairs 1000000 \
  --repetitions 5 \
  --seed 42

## All-vs-all Mode Benchmark

In [ ]:
!python benchmarks/run_real_dataset_benchmark.py \
  --window-size 128 \
  --stride 32 \
  --pairing-mode all_vs_all \
  --max-pairs 1000000 \
  --repetitions 5

## Mutated Queries Mode Benchmark

In [ ]:
!python benchmarks/run_real_dataset_benchmark.py \
  --window-size 128 \
  --stride 32 \
  --pairing-mode mutated_queries \
  --pairs-per-fragment 4 \
  --mutation-rate 0.05 \
  --max-pairs 1000000 \
  --repetitions 5 \
  --seed 42

## Multi-mode Benchmark

In [ ]:
!python benchmarks/run_real_dataset_pairing_modes_benchmark.py

## Optimized Encoded Pipeline

In [ ]:
!nvcc src/hamming_gpu_encoded_optimized.cu -O3 -std=c++17 -I src/common -o hamming_gpu_encoded_optimized

In [ ]:
!./hamming_gpu_encoded_optimized \
  data/processed/sars_cov_2_pairs_128_stride_32_sampled.txt \
  results/hamming/hamming_gpu_encoded_optimized_sampled_results.csv \
  --repetitions 5 \
  --summary-only

In [ ]:
!python benchmarks/run_encoded_optimized_benchmark.py

## Indexed CUDA Graphs Pipeline

In [ ]:
!nvcc src/hamming_gpu_encoded_indexed_graphs.cu \
  -O3 \
  -std=c++17 \
  -I src/common \
  -o hamming_gpu_encoded_indexed_graphs

In [ ]:
!./hamming_gpu_encoded_indexed_graphs \
  data/processed/sars_cov_2_pairs_128_stride_32_all_vs_all.txt \
  results/hamming/hamming_gpu_encoded_indexed_graphs_all_vs_all_results.csv \
  --repetitions 5 \
  --summary-only

In [ ]:
!python benchmarks/run_indexed_graphs_benchmark.py

## Plot Generation

In [ ]:
!python scripts/plot_real_dataset_benchmark.py
!python scripts/plot_real_dataset_pairing_modes.py
!python scripts/plot_encoded_timing_breakdown.py
!python scripts/plot_encoded_optimized_benchmark.py
!python scripts/plot_indexed_graphs_benchmark.py

## Generated Files

In [ ]:
!ls -R benchmarks assets/benchmark_charts/real_dataset_pairing_modes assets/benchmark_charts/encoded_timing_breakdown assets/benchmark_charts/encoded_optimized assets/benchmark_charts/indexed_graphs